# S10 - Series de tiempo e inferencia en streaming: volumen de tráfico del campus (Proyecto Sello)

Actividad autónoma de S10 (4.1) sobre la fuente real de mi Proyecto Sello: los ~400 000 flujos que Suricata capturó en el
campus (`data/TRCU.csv`, 01/07-01/09/2026). Es mi dimensión U2: **¿qué volumen de tráfico (bytes/s) tendrá la red en los
próximos 5 minutos?**

| Tarea 4.1 | Sección |
|---|---|
| 1. `label` con `lead()` sobre el orden cronológico | Fase 3 |
| 2. División cronológica | Fase 3 |
| 3. Baseline de persistencia antes de cualquier modelo | Fase 4.1 |
| 4. Dos o más algoritmos de MLlib contra el baseline | Fase 4.2 - 5.1 |
| 5. Selección honesta del ganador | Fase 5.2 |
| 6. `PipelineModel` guardado y recargado | Fase 6 |
| 7. Inferencia en tiempo real sobre mi propio stream | Fase 7 (`trafico-ventanas`, productor `uso-trafico/ventanas-red-py`) |

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s10-series-tiempo-trafico-campus")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-175ca02b-8132-4b24-9285-33a9b079649e;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central


	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 257ms :: artifacts dl 7ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central i

26/10/22 03:51:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


## Fase 1 — Business Understanding

**Objetivo:** estimar el volumen de tráfico del campus en la **próxima ventana de 5 minutos** (`bytesPorSeg` en t+1) a
partir de lo observado hasta la ventana actual. Es la "proyección de corto plazo" de mi dimensión U2 del brief: anticipar
picos de carga antes de que afecten a la red.

**Alcance:** baseline de persistencia, `LinearRegression`, `RandomForestRegressor` y `GBTRegressor`, con división
cronológica e hiperparámetros elegidos en un tramo de validación, también cronológico. Se reporta RMSE/MAE/R² contra el
baseline, y el ganador se aplica en tiempo real sobre el topic `trafico-ventanas`.

**Decisiones para no inflar el resultado** (se comprueban con datos en la Fase 3):
- Un flujo se ubica en el momento en que **se cierra** (inicio + duración): ahí es cuando Suricata emite su registro y el
  pipeline lo conoce. Repartir sus bytes a lo largo de la duración usaría información que en tiempo real todavía no existe.
- Solo cuentan las ventanas **dentro de una sesión de captura**. Un hueco sin captura no es "tráfico cero", es "sensor
  apagado", y predecir ceros contra ceros infla cualquier métrica.
- Variable objetivo en escala `log1p` (el tráfico tiene ráfagas de varios órdenes de magnitud). Las métricas se reportan
  en esa escala, y el MAE también en bytes/s.

## Fase 2 — Data Understanding

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

RUTA_FLUJOS = "./data/TRCU.csv"
flujos_crudo = (spark.read.csv(RUTA_FLUJOS, header=True)
    .select(F.col("timestamp").cast("long").alias("inicioUs"), F.col("flow_duration").cast("long").alias("duracionUs"),
            (F.col("fwd_pkt_len_tot").cast("double") + F.col("bwd_pkt_len_tot").cast("double")).alias("bytes"),
            F.col("ip_prot").cast("int").alias("protocolo"), F.col("dst_port").cast("int").alias("puertoDestino")))
print(f"Flujos capturados: {flujos_crudo.count():,}")
(flujos_crudo.agg(F.expr("timestamp_micros(min(inicioUs))").alias("primerFlujo"),
                  F.expr("timestamp_micros(max(inicioUs))").alias("ultimoFlujo"),
                  F.expr("percentile_approx(duracionUs, 0.95) / 1e6").alias("duracion_p95_s"),
                  F.expr("max(duracionUs) / 1e6").alias("duracion_max_s"))
             .show(truncate=False))

Flujos capturados: 397,354


+-------------------------+--------------------------+--------------+--------------+
|primerFlujo              |ultimoFlujo               |duracion_p95_s|duracion_max_s|
+-------------------------+--------------------------+--------------+--------------+
|2026-07-01 14:59:38.00385|2026-09-01 10:54:00.442411|100.02706     |119.999999    |
+-------------------------+--------------------------+--------------+--------------+



## Fase 3 — Data Preparation

### 3.1 Flujos ubicados en su cierre y sesiones de captura

In [3]:
GAP_SESION_S = 600          # 10 min sin ningún flujo = el sensor dejó de capturar
MIN_FLUJOS_SESION = 500

orden_cierre = Window.orderBy("cierre")
flujos = (flujos_crudo
    .withColumn("cierre", F.expr("timestamp_micros(inicioUs + duracionUs)"))
    .withColumn("tcp", (F.col("protocolo") == 6).cast("int"))
    .withColumn("web", F.col("puertoDestino").isin(80, 443).cast("int"))
    .withColumn("gapS", F.col("cierre").cast("double") - F.lag(F.col("cierre").cast("double")).over(orden_cierre))
    .withColumn("sesion", F.sum((F.col("gapS") > GAP_SESION_S).cast("int")).over(
        orden_cierre.rowsBetween(Window.unboundedPreceding, Window.currentRow))))
flujos = flujos.withColumn("sesion", F.coalesce("sesion", F.lit(0)))
tam = flujos.groupBy("sesion").count().filter(F.col("count") >= MIN_FLUJOS_SESION)
flujos = flujos.join(tam.select("sesion"), "sesion").cache()

(flujos.groupBy("sesion").agg(F.count("*").alias("flujos"), F.min("cierre").alias("desde"), F.max("cierre").alias("hasta"),
                              F.round((F.max("cierre").cast("double") - F.min("cierre").cast("double")) / 3600, 1).alias("horas"))
       .orderBy("sesion").show(20, truncate=False))

+------+------+--------------------------+--------------------------+-----+
|sesion|flujos|desde                     |hasta                     |horas|
+------+------+--------------------------+--------------------------+-----+
|0     |39622 |2026-07-01 14:59:38.026553|2026-07-03 15:53:34.053481|48.9 |
|2     |9687  |2026-07-03 17:01:18.721185|2026-07-04 18:25:52.579958|25.4 |
|4     |93332 |2026-07-07 08:52:41.422942|2026-07-15 09:46:56.928674|192.9|
|5     |32101 |2026-07-15 10:07:00.087615|2026-07-17 14:42:57.657552|52.6 |
|6     |24466 |2026-07-22 08:51:05.219991|2026-07-24 13:38:21.534145|52.8 |
|7     |56573 |2026-07-30 09:10:58.181005|2026-08-03 17:43:06.140923|104.5|
|8     |57816 |2026-08-07 12:03:22.743789|2026-08-12 07:21:10.581527|115.3|
|9     |59150 |2026-08-13 11:01:54.60173 |2026-08-18 11:28:19.515382|120.4|
|11    |8039  |2026-08-24 08:37:47.691069|2026-08-24 17:59:40.870324|9.4  |
|12    |955   |2026-08-25 10:34:14.301476|2026-08-25 12:03:07.171288|1.5  |
|13    |3671

### 3.2 Serie de 5 minutos por sesión

In [4]:
serie = (flujos.groupBy("sesion", F.window("cierre", "5 minutes").alias("w"))
    .agg(F.sum("bytes").alias("bytes"), F.count("*").alias("flujos"),
         F.avg("tcp").alias("tcpRatio"), F.avg("web").alias("webRatio"))
    .select("sesion", F.col("w.start").alias("ventana"), "bytes", "flujos", "tcpRatio", "webRatio")
    .withColumn("bytesPorSeg", F.col("bytes") / 300)
    .withColumn("ly", F.log1p("bytesPorSeg"))
    .withColumn("lf", F.log1p("flujos")))
print(f"Ventanas de 5 min dentro de sesiones: {serie.count():,}")
serie.select(F.expr("percentile_approx(bytesPorSeg, array(0.5, 0.9, 0.99))").alias("bytesPorSeg_p50_p90_p99"),
             F.max("bytesPorSeg").alias("max"), F.avg("flujos").alias("flujos_prom")).show(truncate=False)

Ventanas de 5 min dentro de sesiones: 8,875


+------------------------------------------------------------+-----------------+-----------------+
|bytesPorSeg_p50_p90_p99                                     |max              |flujos_prom      |
+------------------------------------------------------------+-----------------+-----------------+
|[23.373333333333335, 172.15666666666667, 2772.0666666666666]|52894.84333333333|44.74039436619719|
+------------------------------------------------------------+-----------------+-----------------+



### 3.3 Por qué 5 minutos, flujos cerrados y solo sesiones: el R² de la persistencia en cada alternativa

Si una construcción de la serie hace que "no hacer nada" ya parezca excelente, cualquier modelo encima va a parecer
bueno sin serlo.

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

def r2_persistencia(df, particion):
    w = Window.partitionBy(*particion).orderBy("ventana") if particion else Window.orderBy("ventana")
    d = df.withColumn("label", F.lead("ly").over(w)).na.drop(subset=["label"]).withColumn("prediction", F.col("ly"))
    return RegressionEvaluator(metricName="r2").evaluate(d), d.count()

# (a) toda la línea de tiempo, con los huecos sin captura como "0 bytes"
rango = flujos.agg(F.min("cierre").alias("a"), F.max("cierre").alias("b")).first()
todas = (spark.sql(f"SELECT explode(sequence(timestamp'{rango.a:%Y-%m-%d %H:%M}:00', timestamp'{rango.b:%Y-%m-%d %H:%M}:00', interval 5 minutes)) AS t")
         .withColumn("ventana", F.expr("window(t, '5 minutes').start")).select("ventana").distinct())
con_huecos = (todas.join(serie.select("ventana", "bytesPorSeg"), "ventana", "left").fillna(0, ["bytesPorSeg"])
              .withColumn("ly", F.log1p("bytesPorSeg")))
# (b) 1 minuto dentro de sesiones
serie_1m = (flujos.groupBy("sesion", F.window("cierre", "1 minute").alias("w")).agg(F.sum("bytes").alias("b"))
            .select("sesion", F.col("w.start").alias("ventana"), F.log1p(F.col("b") / 60).alias("ly")))

for nombre, df, part in [("5 min, con huecos sin captura como 0", con_huecos, None),
                         ("1 min, solo sesiones", serie_1m, ["sesion"]),
                         ("5 min, solo sesiones (elegida)", serie, ["sesion"])]:
    r2, n = r2_persistencia(df, part)
    print(f"{nombre:40s} filas={n:7,}  R2 persistencia = {r2:.3f}")

5 min, con huecos sin captura como 0     filas= 17,807  R2 persistencia = 0.944


1 min, solo sesiones                     filas= 37,854  R2 persistencia = -0.161


5 min, solo sesiones (elegida)           filas=  8,861  R2 persistencia = 0.657


### 3.4 Variables (solo pasado y presente) y `label = lead(ly, 1)`

In [6]:
ws = Window.partitionBy("sesion").orderBy("ventana")
VARIABLES = ["l1", "l2", "l3", "l6", "l12", "m6", "m12", "sd6", "f1", "f2", "f3", "mf6",
             "tcpRatio", "webRatio", "hSin", "hCos", "finde"]

dataset_ml = (serie
    .withColumn("rn", F.row_number().over(ws))
    .withColumn("l1", F.col("ly"))
    .withColumn("l2", F.lag("ly", 1).over(ws))
    .withColumn("l3", F.lag("ly", 2).over(ws))
    .withColumn("l6", F.lag("ly", 5).over(ws))
    .withColumn("l12", F.lag("ly", 11).over(ws))
    .withColumn("m6", F.avg("ly").over(ws.rowsBetween(-5, 0)))
    .withColumn("m12", F.avg("ly").over(ws.rowsBetween(-11, 0)))
    .withColumn("sd6", F.stddev("ly").over(ws.rowsBetween(-5, 0)))
    .withColumn("f1", F.col("lf"))
    .withColumn("f2", F.lag("lf", 1).over(ws))
    .withColumn("f3", F.lag("lf", 2).over(ws))
    .withColumn("mf6", F.avg("lf").over(ws.rowsBetween(-5, 0)))
    .withColumn("hSin", F.sin(2 * 3.141592653589793 * F.hour("ventana") / 24))
    .withColumn("hCos", F.cos(2 * 3.141592653589793 * F.hour("ventana") / 24))
    .withColumn("finde", F.dayofweek("ventana").isin(1, 7).cast("int"))
    # label: el volumen de la ventana SIGUIENTE de la misma sesión (nunca cruza un hueco de captura)
    .withColumn("label", F.lead("ly", 1).over(ws))
    .filter(F.col("rn") >= 12)            # historia completa de 12 ventanas
    .select("sesion", "ventana", "bytesPorSeg", *VARIABLES, "label")
    .na.drop()
    .cache())
print(f"Dataset supervisado: {dataset_ml.count():,} filas, {len(VARIABLES)} variables")
dataset_ml.select("ventana", "bytesPorSeg", "l1", "l2", "m6", "f1", "label").orderBy("ventana").show(5, truncate=False)

Dataset supervisado: 8,707 filas, 17 variables
+-------------------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|ventana            |bytesPorSeg       |l1               |l2               |m6               |f1               |label            |
+-------------------+------------------+-----------------+-----------------+-----------------+-----------------+-----------------+
|2026-07-01 15:50:00|4454.5            |8.401894567052775|7.289357929825626|8.172717708482324|4.882801922586371|7.910774014366116|
|2026-07-01 15:55:00|2725.5            |7.910774014366116|8.401894567052775|8.172877534056388|4.51085950651685 |5.84268414372072 |
|2026-07-01 16:00:00|343.7033333333333 |5.84268414372072 |7.910774014366116|7.33397821820489 |4.820281565605037|8.40622811059409 |
|2026-07-01 16:05:00|4473.85           |8.40622811059409 |5.84268414372072 |7.594758268242134|4.875197323201151|7.142381979182829|
|2026-07-01 16:10:00|1263.4366666666

### 3.5 División cronológica: 70 % entrenamiento, 10 % validación (para elegir hiperparámetros), 20 % prueba

In [7]:
numerado = dataset_ml.withColumn("pos", F.row_number().over(Window.orderBy("ventana")))
total = numerado.count()
corte_val, corte_test = int(total * 0.70), int(total * 0.80)

train = numerado.filter(F.col("pos") <= corte_val).drop("pos").cache()
valid = numerado.filter((F.col("pos") > corte_val) & (F.col("pos") <= corte_test)).drop("pos").cache()
train_full = numerado.filter(F.col("pos") <= corte_test).drop("pos").cache()      # train + valid
test = numerado.filter(F.col("pos") > corte_test).drop("pos").cache()

for a, b, na, nb in [(train, valid, "train", "valid"), (valid, test, "valid", "test")]:
    fin_a, ini_b = a.agg(F.max("ventana")).first()[0], b.agg(F.min("ventana")).first()[0]
    if fin_a >= ini_b:
        raise ValueError(f"{na} y {nb} se solapan en el tiempo")
    print(f"{na:5s} hasta {fin_a}  <  {nb} desde {ini_b}  OK")
print(f"\ntrain={train.count():,}  valid={valid.count():,}  test={test.count():,}  (train+valid={train_full.count():,})")

train hasta 2026-08-09 06:15:00  <  valid desde 2026-08-09 06:20:00  OK
valid hasta 2026-08-12 06:50:00  <  test desde 2026-08-12 06:55:00  OK



train=6,094  valid=871  test=1,742  (train+valid=6,965)


## Fase 4 — Modeling

### 4.1 Baseline de persistencia (antes de entrenar nada)

In [8]:
import pandas as pd

ev = {m: RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName=m) for m in ("rmse", "mae", "r2")}

def evaluar(pred, nombre):
    r = {"Modelo": nombre, "RMSE": ev["rmse"].evaluate(pred), "MAE": ev["mae"].evaluate(pred), "R2": ev["r2"].evaluate(pred)}
    # MAE en la escala original (bytes/s), para leerlo sin logaritmos
    r["MAE_Bps"] = pred.select(F.avg(F.abs(F.expm1("label") - F.expm1("prediction")))).first()[0]
    print(f"{nombre:28s} RMSE={r['RMSE']:.4f}  MAE={r['MAE']:.4f}  R2={r['R2']:.4f}  MAE={r['MAE_Bps']:.1f} B/s")
    return r

resultado_persistencia = evaluar(test.withColumn("prediction", F.col("l1")), "Persistencia (y_t+1 = y_t)")
_ = evaluar(valid.withColumn("prediction", F.col("l1")), "Persistencia [validación]")

Persistencia (y_t+1 = y_t)   RMSE=0.6972  MAE=0.2976  R2=0.6652  MAE=196.3 B/s
Persistencia [validación]    RMSE=0.7968  MAE=0.3401  R2=0.6731  MAE=465.5 B/s


### 4.2 Hiperparámetros elegidos en validación (la prueba no se toca)

In [9]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor

ensamblador = VectorAssembler(inputCols=VARIABLES, outputCol="features")
tr_ml, va_ml = ensamblador.transform(train), ensamblador.transform(valid)

grillas = {
    "LinearRegression": [LinearRegression(regParam=r, elasticNetParam=e) for r in (0.0, 0.01, 0.05) for e in (0.0, 0.5)],
    "RandomForestRegressor": [RandomForestRegressor(numTrees=n, maxDepth=d, minInstancesPerNode=10, seed=42)
                              for n in (60, 120) for d in (6, 8, 10)],
    "GBTRegressor": [GBTRegressor(maxIter=i, maxDepth=d, stepSize=0.05, seed=42) for i in (100, 200) for d in (3, 4)],
}
elegidos, filas_val = {}, []
for nombre, candidatos in grillas.items():
    mejor = None
    for est in candidatos:
        rmse = ev["rmse"].evaluate(est.fit(tr_ml).transform(va_ml))
        params = {p.name: v for p, v in est.extractParamMap().items()
                  if p.name in ("regParam", "elasticNetParam", "numTrees", "maxDepth", "maxIter")}
        filas_val.append({"Algoritmo": nombre, "Parámetros": params, "RMSE_valid": round(rmse, 4)})
        if mejor is None or rmse < mejor[0]:
            mejor = (rmse, est)
    elegidos[nombre] = mejor
    print(f"{nombre:22s} mejor RMSE en validación = {mejor[0]:.4f}")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
print(pd.DataFrame(filas_val).to_string(index=False))

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


LinearRegression       mejor RMSE en validación = 0.6646


RandomForestRegressor  mejor RMSE en validación = 0.6668


GBTRegressor           mejor RMSE en validación = 0.6706
            Algoritmo                                                 Parámetros  RMSE_valid
     LinearRegression  {'elasticNetParam': 0.0, 'maxIter': 100, 'regParam': 0.0}      0.6646
     LinearRegression  {'elasticNetParam': 0.5, 'maxIter': 100, 'regParam': 0.0}      0.6646
     LinearRegression {'elasticNetParam': 0.0, 'maxIter': 100, 'regParam': 0.01}      0.6657
     LinearRegression {'elasticNetParam': 0.5, 'maxIter': 100, 'regParam': 0.01}      0.6686
     LinearRegression {'elasticNetParam': 0.0, 'maxIter': 100, 'regParam': 0.05}      0.6683
     LinearRegression {'elasticNetParam': 0.5, 'maxIter': 100, 'regParam': 0.05}      0.6743
RandomForestRegressor                            {'maxDepth': 6, 'numTrees': 60}      0.6729
RandomForestRegressor                            {'maxDepth': 8, 'numTrees': 60}      0.6686
RandomForestRegressor                           {'maxDepth': 10, 'numTrees': 60}      0.6677
RandomForestR

### 4.3 Reentrenar con train + valid y evaluar UNA vez sobre la prueba

In [10]:
trf_ml, te_ml = ensamblador.transform(train_full), ensamblador.transform(test)
modelos, resultados = {}, [resultado_persistencia]
for nombre, (_, est) in elegidos.items():
    modelos[nombre] = est.copy().fit(trf_ml)
    resultados.append(evaluar(modelos[nombre].transform(te_ml), nombre))

lr_m = modelos["LinearRegression"]
print("\nLinearRegression - coeficientes:")
for v, c in sorted(zip(VARIABLES, lr_m.coefficients.toArray()), key=lambda x: -abs(x[1])):
    print(f"   {v:9s} {c:+.4f}")
print(f"   intercepto {lr_m.intercept:+.4f}")
print("\nRandomForest - importancia de variables:")
for v, i in sorted(zip(VARIABLES, modelos["RandomForestRegressor"].featureImportances.toArray()), key=lambda x: -x[1])[:8]:
    print(f"   {v:9s} {i:.4f}")

LinearRegression             RMSE=0.5896  MAE=0.2771  R2=0.7606  MAE=145.8 B/s


RandomForestRegressor        RMSE=0.5862  MAE=0.3000  R2=0.7634  MAE=147.4 B/s


GBTRegressor                 RMSE=0.5968  MAE=0.2926  R2=0.7547  MAE=149.0 B/s

LinearRegression - coeficientes:
   mf6       -0.6835
   tcpRatio  +0.6072
   f1        +0.4899
   m6        +0.4329
   f3        +0.2752
   m12       +0.2198
   f2        +0.2083
   l1        +0.1663
   sd6       -0.1353
   l2        +0.0838
   l6        -0.0768
   webRatio  -0.0738
   hCos      -0.0663
   finde     -0.0618
   hSin      +0.0276
   l3        +0.0058
   l12       -0.0037
   intercepto -0.4093

RandomForest - importancia de variables:
   m6        0.2763
   l1        0.2466
   l2        0.1636
   f1        0.0883
   mf6       0.0737
   f2        0.0279
   tcpRatio  0.0236
   l3        0.0234


## Fase 5 — Evaluation

### 5.1 Comparación contra el baseline

In [11]:
comparacion = pd.DataFrame(resultados)
base = resultado_persistencia
comparacion["Mejora_RMSE_%"] = 100 * (base["RMSE"] - comparacion["RMSE"]) / base["RMSE"]
comparacion["Mejora_MAE_%"] = 100 * (base["MAE"] - comparacion["MAE"]) / base["MAE"]
print(comparacion.round(4).to_string(index=False))

                    Modelo   RMSE    MAE     R2  MAE_Bps  Mejora_RMSE_%  Mejora_MAE_%
Persistencia (y_t+1 = y_t) 0.6972 0.2976 0.6652 196.2724         0.0000        0.0000
          LinearRegression 0.5896 0.2771 0.7606 145.7736        15.4293        6.8859
     RandomForestRegressor 0.5862 0.3000 0.7634 147.3667        15.9206       -0.7857
              GBTRegressor 0.5968 0.2926 0.7547 148.9703        14.3902        1.7022


### 5.2 Selección honesta del ganador

In [12]:
# Regla fijada ANTES de mirar la prueba: gana el mejor RMSE de validación; si otro algoritmo
# más simple queda a menos de 1 % de ese RMSE, se elige el más simple (parsimonia).
SIMPLICIDAD = ["LinearRegression", "RandomForestRegressor", "GBTRegressor"]
mejor_val = min(elegidos, key=lambda n: elegidos[n][0])
ganador = next(n for n in SIMPLICIDAD if elegidos[n][0] <= elegidos[mejor_val][0] * 1.01)
print(f"Mejor en validación: {mejor_val} ({elegidos[mejor_val][0]:.4f}); ganador por la regla: {ganador} ({elegidos[ganador][0]:.4f})")

fila = comparacion.set_index("Modelo").loc[ganador]
print(f"En la prueba: RMSE {fila['RMSE']:.4f} frente a {base['RMSE']:.4f} de la persistencia "
      f"({fila['Mejora_RMSE_%']:+.1f} %), R2 {fila['R2']:.3f} frente a {base['R2']:.3f}, "
      f"MAE {fila['MAE_Bps']:.1f} frente a {base['MAE_Bps']:.1f} B/s")

Mejor en validación: LinearRegression (0.6646); ganador por la regla: LinearRegression (0.6646)
En la prueba: RMSE 0.5896 frente a 0.6972 de la persistencia (+15.4 %), R2 0.761 frente a 0.665, MAE 145.8 frente a 196.3 B/s


## Fase 6 — Deployment: `PipelineModel` guardado y recargado

In [13]:
from pyspark.ml import Pipeline, PipelineModel

RUTA_MODELO = "./artifacts/models/modelo_trafico_5min"
pipeline_ganador = Pipeline(stages=[VectorAssembler(inputCols=VARIABLES, outputCol="features"), elegidos[ganador][1].copy()])
modelo_final = pipeline_ganador.fit(train_full)
modelo_final.write().overwrite().save(RUTA_MODELO)

modelo_cargado = PipelineModel.load(RUTA_MODELO)
print("Etapas del modelo guardado:", [type(s).__name__ for s in modelo_cargado.stages])
antes = modelo_final.transform(test).select("ventana", F.col("prediction").alias("p_antes"))
despues = modelo_cargado.transform(test).select("ventana", F.col("prediction").alias("p_despues"))
dif = antes.join(despues, "ventana").select(F.max(F.abs(F.col("p_antes") - F.col("p_despues")))).first()[0]
print(f"Predicciones sobre {test.count():,} ventanas de prueba: diferencia máxima antes/después de recargar = {dif}")
evaluar(modelo_cargado.transform(test), f"{ganador} (recargado)")

Etapas del modelo guardado: ['VectorAssembler', 'LinearRegressionModel']


Predicciones sobre 1,742 ventanas de prueba: diferencia máxima antes/después de recargar = 0.0


LinearRegression (recargado) RMSE=0.5896  MAE=0.2771  R2=0.7606  MAE=145.8 B/s


{'Modelo': 'LinearRegression (recargado)',
 'RMSE': 0.58959591482222,
 'MAE': 0.27713601186669357,
 'R2': 0.7605775789601716,
 'MAE_Bps': 145.77361728514794}

## Fase 7 — Inferencia en tiempo real sobre `trafico-ventanas`

El productor `uso-trafico/ventanas-red-py` hace de agregador de borde: lee los flujos reales de un tramo de la **prueba**
(14/08, desde las 08:00), arma cada ventana de 5 min con su historia y publica una ventana cada 2 s. Cada evento ya trae
las 17 variables, así que Spark solo hace `transform()`: sin `groupBy`, sin ventanas y sin estado.

In [14]:
import os, shutil, time
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema = (StructType().add("tipoEvento", StringType()).add("sensorId", StringType())
           .add("ventanaInicio", LongType()).add("bytesPorSeg", DoubleType()).add("flujos", LongType()))
for v in VARIABLES:
    esquema = esquema.add(v, DoubleType())
esquema = esquema.add("origen", StringType()).add("timestamp", LongType())

eventos = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", "kafka:9092")
           .option("subscribe", "trafico-ventanas").option("startingOffsets", "earliest")
           .option("failOnDataLoss", "false").load()
           .select(F.from_json(F.col("value").cast("string"), esquema).alias("e")).select("e.*")
           .withColumn("ventana", F.timestamp_millis("ventanaInicio")))

predicciones_stream = modelo_cargado.transform(eventos)

RUTA_PRED = "./artifacts/predicciones_stream"
CHK_PRED = "./artifacts/chk_predicciones_stream"
shutil.rmtree(RUTA_PRED, ignore_errors=True); shutil.rmtree(CHK_PRED, ignore_errors=True)
bitacora = []

def guardar_microlote(df, batch_id):
    filas = df.select(F.lit(batch_id).alias("batchId"), "ventana", "bytesPorSeg", "flujos",
                      F.expm1("prediction").alias("prediccionSiguienteBps"), "prediction",
                      F.col("timestamp").alias("publicadoMs"),
                      F.unix_millis(F.current_timestamp()).alias("procesadoMs"))
    n = filas.count()
    if n == 0:
        bitacora.append(f"Micro-lote {batch_id}: sin filas")
        return
    filas.write.mode("append").parquet(RUTA_PRED)
    bitacora.append(f"Micro-lote {batch_id}: {n} predicciones")

consulta = (predicciones_stream.writeStream.foreachBatch(guardar_microlote)
            .option("checkpointLocation", CHK_PRED).trigger(processingTime="3 seconds").start())
inicio = time.time()
while time.time() - inicio < 420:
    time.sleep(15)
    hechas = spark.read.parquet(RUTA_PRED).count() if any(f.endswith(".parquet") for _, _, fs in os.walk(RUTA_PRED) for f in fs) else 0
    if hechas >= 120:
        break
consulta.stop()
print("\n".join(bitacora[:6] + ["..."] + bitacora[-3:]))
print(f"Consulta detenida tras {time.time() - inicio:.0f} s")

Micro-lote 0: 1 predicciones
Micro-lote 1: 1 predicciones
Micro-lote 2: 2 predicciones
Micro-lote 3: 1 predicciones
Micro-lote 4: 1 predicciones
Micro-lote 5: 2 predicciones
...
Micro-lote 78: 1 predicciones
Micro-lote 79: 2 predicciones
Micro-lote 80: 1 predicciones
Consulta detenida tras 248 s


In [15]:
pred = spark.read.parquet(RUTA_PRED)
w_t = Window.orderBy("ventana")
resultado = (pred.withColumn("realSiguienteBps", F.lead("bytesPorSeg").over(w_t))
                 .withColumn("latenciaMs", F.col("procesadoMs") - F.col("publicadoMs")))
print(f"Predicciones en tiempo real: {pred.count()}  |  latencia publicación -> predicción: "
      f"p50 {resultado.select(F.expr('percentile_approx(latenciaMs, 0.5)')).first()[0]} ms")
(resultado.select("batchId", F.date_format("ventana", "dd/MM HH:mm").alias("ventana"),
                  F.round("bytesPorSeg", 1).alias("realBps"), F.round("prediccionSiguienteBps", 1).alias("predSiguienteBps"),
                  F.round("realSiguienteBps", 1).alias("realSiguienteBps"))
          .orderBy("ventana").show(12, truncate=False))

# Las mismas ventanas en la evaluación fuera de línea: el stream debe predecir exactamente lo mismo
offline = modelo_cargado.transform(test).select("ventana", F.col("prediction").alias("p_offline"))
par = pred.join(offline, "ventana")
print(f"Ventanas del stream encontradas en la prueba offline: {par.count()} de {pred.count()}; "
      f"diferencia máxima de predicción: {par.select(F.max(F.abs(F.col('prediction') - F.col('p_offline')))).first()[0]:.2e}")

eval_stream = (resultado.na.drop(subset=["realSiguienteBps"])
               .withColumn("label", F.log1p("realSiguienteBps")))
r_m = evaluar(eval_stream, f"{ganador} en el stream")
r_p = evaluar(eval_stream.withColumn("prediction", F.log1p("bytesPorSeg")), "Persistencia en el stream")
resultado.orderBy("ventana").toPandas().to_csv("./artifacts/predicciones_stream.csv", index=False)

Predicciones en tiempo real: 120  |  latencia publicación -> predicción: p50 2026 ms


+-------+-----------+-------+----------------+----------------+
|batchId|ventana    |realBps|predSiguienteBps|realSiguienteBps|
+-------+-----------+-------+----------------+----------------+
|0      |14/08 08:00|23.0   |26.8            |31.5            |
|1      |14/08 08:05|31.5   |30.7            |22.4            |
|2      |14/08 08:10|22.4   |27.6            |25.9            |
|2      |14/08 08:15|25.9   |28.3            |29.5            |
|3      |14/08 08:20|29.5   |31.8            |77.0            |
|4      |14/08 08:25|77.0   |43.8            |191.7           |
|5      |14/08 08:30|191.7  |80.4            |102.3           |
|5      |14/08 08:35|102.3  |80.2            |115.3           |
|6      |14/08 08:40|115.3  |92.8            |88.5            |
|7      |14/08 08:45|88.5   |75.1            |77.2            |
|7      |14/08 08:50|77.2   |72.4            |75.2            |
|8      |14/08 08:55|75.2   |65.8            |74.0            |
+-------+-----------+-------+-----------

Ventanas del stream encontradas en la prueba offline: 120 de 120; diferencia máxima de predicción: 8.56e-07


LinearRegression en el stream RMSE=0.5144  MAE=0.3541  R2=0.4385  MAE=24.0 B/s


Persistencia en el stream    RMSE=0.6461  MAE=0.4173  R2=0.1140  MAE=30.7 B/s


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
